# Power-transform ELI data

## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar
import scipy.optimize
from sklearn.preprocessing import PowerTransformer

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## RNG
rng = np.random.default_rng()

## Loading

### Load indices

In [ ]:
## open data
Th = src.utils.load_cesm_indices()

## omit first year (bc of NaN in h,hw vars)
Th = Th.sel(time=slice("1851", None))

## Load ELI
eli = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/eli.nc"))
eli_forced, eli_anom = src.utils.separate_forced(eli)

## merge
Th = xr.merge([Th, eli_anom.sel(time=slice("1851", None))])

## scale for convenience
Th_scale = Th.std()
Th /= Th_scale

## get windowed
Th = src.utils.get_windowed(Th, window_size=480, stride=120)

### Scale by $dT/dx$

In [ ]:
## load dT/dx data
dTdx = xr.open_dataarray(pathlib.Path(SAVE_FP, "cesm_dTdx_Tw.nc"))
# dTdx = xr.open_dataarray(pathlib.Path(SAVE_FP, "cesm_dTdx.nc"))

## scale by initial value
dTdx_scale = dTdx / dTdx.isel(year=0).mean(["month"])

for v in list(Th):
    if "eli" in v:
        Th[f"{v}_scaled"] = Th[v].groupby("time.month") * dTdx_scale

        ## subtrack median
        grouped = Th[f"{v}_scaled"].groupby("time.month")
        Th[f"{v}_scaled_median"] = grouped - grouped.median(["member", "time"])


# Th["eli_05_scaled"].to_netcdf(SAVE_FP / "eli_scaled_Tw.nc")

## Transform

### funcs

In [ ]:
def fit_pt(z, get_lambda=False):
    """fit power transform to data"""
    if np.isnan(z).all():
        return np.nan

    else:
        ## fit transform
        pt = PowerTransformer(standardize=True).fit(z[:, None])

        if get_lambda:
            return pt, pt.lambdas_.item()

        else:
            return pt


def fit_transform_pt(X):
    """fit power transform to data and return Xarray"""

    ## empty xarray to hold data
    dims = copy.deepcopy(X.dims)
    X_stack = X.stack(s=dims)
    Y_stack = copy.deepcopy(X_stack)

    ## do transform
    pt, lam = fit_pt(X_stack.values, get_lambda=True)
    Y_stack.values = (
        pt.transform(X_stack.values[:, None]).flatten() * X_stack.values.std()
    )

    return Y_stack.unstack("s")

### do the transform

In [ ]:
## select data
V = "eli_05_scaled"
X = Th[V]

## load/save to file
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"], f"{V}_pt_Tw_updated.nc")
if SAVE_FP.is_file():
    Y = xr.open_dataarray(SAVE_FP)

else:

    # ## transform data each year separately
    # Y = []
    # for year in X.year:
    #     Y.append(X.sel(year=year).groupby("time.month").map(fit_transform_pt))
    # Y = xr.concat(Y, dim=X.year)

    ## transform data for all years at the same time
    Y = X.groupby("time.month").map(fit_transform_pt)

    ## save
    Y.to_netcdf(SAVE_FP)

### Plots

#### Plot PDFs

In [ ]:
## funcs to select data
sel = lambda x: src.utils.sel_month(x.sel(year=2031), 1)
sel_ = lambda x: sel(x).stack(s=["time", "member"])

## plot distributions
bins = np.linspace(-4, 4, 40)
pdf_X, _ = src.utils.get_empirical_pdf(sel(X), bins)
pdf_Y, _ = src.utils.get_empirical_pdf(sel(Y), bins)

fig, ax = plt.subplots(figsize=(4, 3.5), layout="constrained")
ax.stairs(pdf_X, bins)
ax.stairs(pdf_Y, bins)
ax.set_xlim([-3, 3])
ax.axvline(0, c="k", lw=1)
plt.show()

#### Plot relation to Niño 3

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.5), layout="constrained")

## plot
plot_kwargs = dict(s=3, alpha=0.5)
axs[0].scatter(sel(X), sel(Th["T_3"]), **plot_kwargs)
axs[1].scatter(sel(Y), sel(Th["T_3"]), **plot_kwargs)

## align axes
axs[1].set_ylim(axs[0].get_ylim())
axs[1].set_yticks([])

## add axes
ax_kwargs = dict(c="k", lw=0.8)
for ax in axs:
    ax.axvline(0, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)

plt.show()

### Change in stats

In [ ]:
## select data
V = "eli_05_scaled_median"
X = Th[V].sel(year=1871)
Y = Th[V].sel(year=2021)
Z = Th[V].sel(year=2071)
# X = Th[V].sel(year=2021)
# Y = Th[V].sel(year=2081)

## funcs to select data
sel = lambda x: src.utils.sel_month(x, [4])
sel_ = lambda x: sel(x).stack(s=["time", "member"])

## plot distributions
bins = np.linspace(-4, 4, 40)
pdf_X, _ = src.utils.get_empirical_pdf(sel(X), bins)
pdf_Y, _ = src.utils.get_empirical_pdf(sel(Y), bins)
pdf_Z, _ = src.utils.get_empirical_pdf(sel(Z), bins)

fig, ax = plt.subplots(figsize=(4, 3.5), layout="constrained")
ax.stairs(pdf_X, bins)
ax.stairs(pdf_Y, bins)
ax.stairs(pdf_Z, bins)
ax.set_xlim([-1.5, 3.5])
ax.axvline(0, c="k", lw=1)
plt.show()

## ELI coords

In [ ]:
def get_T_helper(y, dTdx):
    """get temperature anomaly as a function of distance given:
    - warm pool posn
    - horizontal temperature gradient
    """

    ## test points
    x = xr.DataArray(
        np.linspace(140, 280, 141),
        coords=dict(lon=np.linspace(140, 280, 141)),
    )

    ## find out whether we're in the cold tongue
    in_ct = (x > y).astype(float)

    return dTdx * (y - x) * in_ct, x


def get_T(y0, yp, dTdx):
    """get temperature anomaly as a function of distance given:
    - warm pool mean state
    - warm pool anomaly
    - horizontal temperature gradient
    """

    ## test points
    x = np.linspace(140, 280)

    ## first, get mean temp
    Tbar, x = get_T_helper(y=y0, dTdx=dTdx)

    ## get total temp
    T, _ = get_T_helper(y=y0 + yp, dTdx=dTdx)

    return T - Tbar, x


def get_func(func, dTdx, y0):
    """compute function of eli"""

    ## get ELI range
    eli_range = xr.DataArray(
        np.arange(140, 280),
        coords=dict(eli_lon=np.arange(140, 280)),
    )

    ## get anomalies
    yp = eli_range - y0

    ## recon Temp
    T, x = get_T(dTdx=dTdx, yp=yp, y0=y0)

    return func(T)

In [ ]:
## get DJF horiz. temp. diff
dT_djf = dTdx.sel(month=[12, 1, 2]).mean("month")

## convert to gradient
dTdx_djf = dT_djf / 140

## get warm pool position
eli_total = (eli_forced + eli_anom).sel(time=slice("1851", None))
y = src.utils.get_windowed(eli_total, stride=120)["eli_05"] + 0
y = src.utils.sel_month(y.resample({"time": "QS-DEC"}).mean(), 12)
y0 = y.median(["time", "member"])

#### Plot a sample

In [ ]:
kwargs = dict(
    yp=50,
    y0=y0.isel(year=0),
    dTdx=dTdx_djf.isel(year=0),
)

T, x = get_T(**kwargs)

fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(x, T)
plt.show()

Helper funcs

In [ ]:
avg_lons = lambda x, lon_range: x.sel(lon=slice(*lon_range)).mean("lon")
get_T3 = lambda x: avg_lons(x, (210, 270))
get_T34 = lambda x: avg_lons(x, (190, 240))
get_T4 = lambda x: avg_lons(x, (160, 210))

Look at effect of changing gradient (holding mean posn constant)

In [ ]:
dTdx_djf

In [ ]:
import pandas as pd

dTdx_vals = np.array([0.028, 0.024, 0.019]) * 2
y0_ = y0.isel(year=0)

T34s = xr.concat(
    [get_func(func=get_T34, y0=y0_, dTdx=d) for d in dTdx_vals],
    dim=pd.Index(dTdx_vals, name="dTdx"),
)

## trim
T34s = T34s.sel(eli_lon=slice(None, 240))

## get scaled x axis
lon_scaled = (T34s.eli_lon - y0_) * T34s.dTdx

In [ ]:
dTdx_vals * 140

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(T34s.eli_lon, T34s.isel(dTdx=0))
ax.plot(T34s.eli_lon, T34s.isel(dTdx=1))
ax.plot(T34s.eli_lon, T34s.isel(dTdx=2))

# ax.plot(lon_scaled.isel(dTdx=0), T34s.isel(dTdx=0))
# ax.plot(lon_scaled.isel(dTdx=1), T34s.isel(dTdx=1))
# ax.plot(lon_scaled.isel(dTdx=2), T34s.isel(dTdx=2))

plt.show()

In [ ]:
dTdx_vals

In [ ]:
H = lambda x: (x > 0).astype(float)


def integral(x, y, dTdx):
    return 0.5 * (-dTdx) * (x - y) ** 2 * H(x - y)


def def_integral(x0, x1, y, dTdx):
    """get def integral"""

    kwargs = dict(y=y, dTdx=dTdx)

    return 1 / (x1 - x0) * (integral(x=x1, **kwargs) - integral(x=x0, **kwargs))


y_vals = np.linspace(140, 240)

## get full
kwargs = dict(x1=240, x0=190)

res = []
for dTdx_ in dTdx_vals:
    res_ = def_integral(y=y_vals, dTdx=dTdx_, **kwargs)
    res_mean = def_integral(y=np.array([160]), dTdx=dTdx_, **kwargs)

    res.append(res_ - res_mean)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(y_vals, res[0])
ax.plot(y_vals, res[1])
ax.plot(y_vals, res[2])
plt.show()

effect of changing mean posn

In [ ]:
import pandas as pd

y0_vals = [160, 170, 180]
dTdx_ = dTdx_djf.isel(year=0)

T34s = xr.concat(
    [get_func(func=get_T34, y0=y0_, dTdx=dTdx_) for y0_ in y0_vals],
    dim=pd.Index(y0_vals, name="y0"),
)

## trim
T34s = T34s.sel(eli_lon=slice(None, 240))

In [ ]:
# norm = lambda x : x-x.mean("eli_lon")
norm = lambda x: x

fig, ax = plt.subplots(figsize=(4, 3))

ax.plot(T34s.eli_lon, norm(T34s.isel(y0=0)))

ax.plot(T34s.eli_lon, norm(T34s.isel(y0=1)))

ax.plot(T34s.eli_lon, norm(T34s.isel(y0=2)))

plt.show()

#### Compare to real data

In [ ]:
kwargs = dict(dTdx=dTdx_djf.isel(year=0), y0=y0)
T3 = get_func(func=get_T3, **kwargs)
T34 = get_func(func=get_T34, **kwargs)
T4 = get_func(func=get_T4, **kwargs)

In [ ]:
get_djf = lambda x: src.utils.sel_month(x.resample({"time": "QS-DEC"}).mean(), 12)
X = get_djf(Th).transpose(*y.dims)


fig, axs = plt.subplots(1, 2, figsize=(7, 3.5), layout="constrained")

# fig,ax = plt.subplots(figsize=(3,2.5))
# axs[0].plot(T4.eli_lon, T4, c="r")
# axs[0].plot(T4.eli_lon, T34.sel(year=1991), c="k")
# axs[0].plot(T4.eli_lon, T3, c="b")

## plot
plot_kwargs = dict(s=3, alpha=0.5)
axs[0].scatter(y.sel(year=1991), X["T_34"].sel(year=1991), **plot_kwargs)
axs[1].scatter(y.sel(year=2071), X["T_34"].sel(year=2071), **plot_kwargs)

## align axes
src.utils.set_lims(axs)
# axs[1].set_ylim(axs[0].get_ylim())
axs[1].set_yticks([])

## add axes
ax_kwargs = dict(c="k", lw=0.8)
for ax in axs:
    ax.axvline(0, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)
    # ax.set_ylim([-4,4])

plt.show()

#### look at some samples

In [ ]:
forced, anom = src.utils.load_consolidated()
sst = xr.merge([forced["sst_comp"], forced["sst"] + anom["sst"]])
sst = sst.sel(time=slice("1851", "1890"))
sst = src.utils.sel_month(sst.resample({"time": "QS-DEC"}).mean(), 12)

## get meridional mdean
merimean = lambda x: x.sel(longitude=slice(120, 280), latitude=slice(-5, 5)).mean(
    "latitude"
)
sst = src.utils.reconstruct_wrapper(sst, fn=merimean)["sst"]
sst["time"] = y.time.values

In [ ]:
idx = dict(member=4, time=10)
fig, ax = plt.subplots(figsize=(4, 3))

ax.plot(sst.longitude, sst.isel(idx))
ax.plot(sst.longitude, sst.mean(["time", "member"]), c="k")

ax.axvline(y.isel(year=0, **idx))
ax.axvline(y.isel(year=0).mean(["time", "member"]), c="k")

plt.show()

In [ ]:
idx = dict(member=4, time=10)
fig, ax = plt.subplots(figsize=(4, 3))

ax.plot(sst.longitude, sst.isel(idx) - sst.mean(["time", "member"]))
# ax.plot(sst.longitude, sst.mean(["time","member"]), c="k")

# ax.axvline(y.isel(year=0, **idx))
# ax.axvline(y.isel(year=0).mean(["time","member"]), c="k")